### CRISP-DM Phase 2 - Data Understanding : Laws & Policies

Exploratory analysis of **Climate Change Laws of the World** by *Climate Policy Radar* and **Climate Policy Database** by *NewClimate Institute*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import ftfy
from iso3166 import countries_by_alpha3
import geopandas as gpd

In [ ]:
# Load the datasets
cclw = pd.read_csv('data/CCLW-27-04-2026.csv')
cpdb = pd.read_csv('data/CPDB-21-04-2026.csv')

df_dict = {'CCLW': cclw, 'CPDB': cpdb}

DU1 - Attributes

In [ ]:
def attributes(dict):
    for name, df in dict.items():
        print(f"Dataset: {name}\nShape: {df.shape}\nColumns: {df.columns.tolist()}")
        print(f"Data types:\n{df.dtypes}")

attributes(df_dict)

DU2 - Statistical properties

In [ ]:
## Target Variable
# Count
print(f"Documents with hazard labels: {cclw['Hazard'].notna().sum()}")

# Distribution of labels
benchmark = cclw[cclw['Hazard'].notna()].copy()
benchmark['Hazard'] = benchmark['Hazard'].str.strip().str.lower()
hazard_count = (benchmark['Hazard'].str.split(';').explode().str.strip().value_counts())
print(f"Number of unique hazard labels: {len(hazard_count.index.tolist())}")
# Force to display all rows
#with pd.option_context('display.max_rows', None): 
#   print(f"Unique hazard label frequency:\n{hazard_count}")

# Plot top 10 hazard labels 
top_hazards = hazard_count.head(10)
plt.figure(figsize=(7,5))
top_hazards.sort_values().plot(kind='barh')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('outputs/2_top_hazards.png', dpi=150, bbox_inches='tight')
plt.close()
    
# Number of labels per document
label_distrib = benchmark['Hazard'].str.split(';').apply(len).value_counts().sort_index()
#print(f"Distribution of number of labels per document:\n{label_distrib}")

# Plot distribution of number of labels per document
plt.figure(figsize=(7,5))
label_distrib.plot(kind='bar')
plt.xlabel('Number of Hazard Labels')
plt.ylabel('Number of Documents')

plt.tight_layout()
plt.savefig('outputs/2_hazard_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Geographic Coverage
cclw_countries = set(cclw['Geography ISOs'].dropna().unique())
cpdb_countries = set(cpdb['country_iso'].dropna().unique())

print(f"Total countries in CCLW: {len(cclw_countries)} and CPDB: {len(cpdb_countries)}")
print(f"Countries in CCLW only: {len(cclw_countries - cpdb_countries)}\n{cclw_countries - cpdb_countries}")
print(f"Countries in CPDB only: {len(cpdb_countries - cclw_countries)}\n{cpdb_countries - cclw_countries}")
print(f"Countries in both: {len(cclw_countries & cpdb_countries)}")

# Count documents with country XAA/XAB with hazard labels
xaa_hazard_count = cclw[(cclw['Geography ISOs'] == 'XAA') & (cclw['Hazard'].notna())].shape[0]
xab_hazard_count = cclw[(cclw['Geography ISOs'] == 'XAB') & (cclw['Hazard'].notna())].shape[0]
print(f"Documents with country XAA and hazard labels: {xaa_hazard_count}")
print(f"Documents with country XAB and hazard labels: {xab_hazard_count}")

# Plot top 10 countries by document count
cclw_top10 = cclw['Geography ISOs'].value_counts().head(10)
cpdb_top10 = cpdb['country_iso'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cclw_top10.plot(kind='barh', ax=axes[0])
axes[0].set_xlabel('Number of documents')
axes[0].set_ylabel('Country ISO')
axes[0].invert_yaxis()
cpdb_top10.plot(kind='barh', ax=axes[1])
axes[1].set_xlabel('Number of documents')
axes[1].set_ylabel('Country ISO')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('outputs/2_top_countries.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Temporal Coverage
# 3 dates wrong : 0021 instead of 2021, 0022 instead of 2022, 1010 instead of 2010 - manually corrected directly in the csv
cclw_year = pd.to_datetime(cclw['First event in timeline'], errors='coerce').dt.year.astype('Int64')
cpdb_year = pd.to_numeric(cpdb['decision_date'], errors='coerce').astype('Int64')

for name, year in [('CCLW', cclw_year), ('CPDB', cpdb_year)]:
    print(f"{name} year range: {year.min()} - {year.max()}")
    #print(f"Documents per decade:\n{year.floordiv(10).mul(10).value_counts().sort_index()}\n")

# Plot per decade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, year) in zip(axes, [('CCLW', cclw_year), ('CPDB', cpdb_year)]):
    decade_counts = year.floordiv(10).mul(10).value_counts().sort_index().dropna()
    ax.bar(decade_counts.index.astype(str), decade_counts.values)
    ax.set_xlabel('Decade')
    ax.set_ylabel('Number of documents')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('outputs/2_decade_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

# Plot per year 2000-2026
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, year) in zip(axes, [('CCLW', cclw_year), ('CPDB', cpdb_year)]):
    yearly = year[(year >= 2000) & (year <= 2026)].value_counts().sort_index()
    ax.bar(yearly.index.astype(str), yearly.values)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of documents')
    ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig('outputs/2_year_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Topic Distribution
cclw_topic = set(cclw['Topic/Response'].dropna().unique())
cpdb_topic = set(cpdb['policy_objective'].dropna().unique())

print(f"Total unique topics in CCLW: {len(cclw_topic)}")
print(f"Topic frequency:\n{cclw['Topic/Response'].str.split(';').explode().str.strip().value_counts()}")
print(f"Total unique topics in CPDB: {len(cpdb_topic)}")
print(f"Topic frequency:\n{cpdb['policy_objective'].str.split(',').explode().str.strip().value_counts()}")

In [ ]:
## Text Length Distribution
# Clean HTML format 
def clean_text(text):
    if pd.isna(text): return ''
    text = ftfy.fix_text(str(text))
    return BeautifulSoup(text, 'html.parser').get_text(separator=' ').strip()

cclw_text = cclw['Family Summary'].apply(clean_text)
cpdb_text = cpdb['policy_description'].apply(clean_text)

def word_count(text):
    return len(text.split()) if text else 0

cclw_length = cclw_text.apply(word_count)
cpdb_length = cpdb_text.apply(word_count)

# Statistics
for name, length in [('CCLW', cclw_length), ('CPDB', cpdb_length)]:
    print(f"\n{name} text length stats (words):")
    print(length.describe().round(0))

# Threshold analysis
thresholds = [5, 15, 25]
for name, length, df in [('CCLW', cclw_length, cclw), ('CPDB', cpdb_length, cpdb)]:
    print(f"\n{name} text length thresholds:")
    for t in thresholds:
        count = (length < t).sum()
        print(f"  Under {t} words: {count} docs ({count/len(df)*100:.1f}%)")

DU3 - Data Quality

In [ ]:
## Missing Values
print(f"CCLW missing values (%):\n{(cclw.isnull().sum() / len(cclw) * 100).round(1)}")
print(f"CPDB missing values (%):\n{(cpdb.isnull().sum() / len(cpdb) * 100).round(1)}")

DU4 - Visual Exploration

In [ ]:
# Map ISO codes to country names
def iso_to_name(iso):
    try:
        result = countries_by_alpha3.get(iso)
        if result is None:
            return None
        return result.name
    except:
        return None

cclw_countries = cclw['Geography ISOs'].str.strip().value_counts().reset_index()
cclw_countries.columns = ['iso', 'count']
cclw_countries['country'] = cclw_countries['iso'].apply(iso_to_name)

cpdb_countries = cpdb['country_iso'].str.strip().value_counts().reset_index()
cpdb_countries.columns = ['iso', 'count']
cpdb_countries['country'] = cpdb_countries['iso'].apply(iso_to_name)

# Merge with world map
world = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
iso_fixes = {'France': 'FRA', 'Norway': 'NOR', 'South Sudan': 'SSD', 'Kosovo': 'XKX', 
             'Northern Cyprus': 'CYN', 'Somaliland': 'SOM'}
for country_name, correct_iso in iso_fixes.items():
    world.loc[world['NAME'] == country_name, 'ISO_A3'] = correct_iso
cclw_merged = world.merge(cclw_countries, left_on='ISO_A3', right_on='iso', how='left')
cpdb_merged = world.merge(cpdb_countries, left_on='ISO_A3', right_on='iso', how='left')

fig, axes = plt.subplots(2, 1, figsize=(15, 12))
for ax, merged, name in [(axes[0], cclw_merged, 'CCLW'), (axes[1], cpdb_merged, 'CPDB')]:
    merged.plot(column='count', ax=ax, legend=True,missing_kwds={'color': 'lightgrey'}, cmap='YlOrRd',
                legend_kwds={'shrink': 0.6, 'label': 'Number of documents'})
    ax.axis('off') 
plt.tight_layout()
plt.savefig('outputs/2_world_map.png', dpi=150, bbox_inches='tight')
plt.close()

DU5 - Ethical Concerns, Risks, and Biase

In [ ]:
## Population Comparison
cclw_pop_correlation = cclw_merged[['count', 'POP_EST']].corr().iloc[0,1]
print(f"Correlation between document count and population in CCLW: {cclw_pop_correlation.round(2)}")

cpdb_pop_correlation = cpdb_merged[['count', 'POP_EST']].corr().iloc[0,1]
print(f"Correlation between document count and population in CPDB: {cpdb_pop_correlation.round(2)}")

In [ ]:
## Country Size Comparison
cclw_merged['area_km2'] = cclw_merged['geometry'].to_crs('ESRI:54009').area / 1e6
cclw_size_correlation = cclw_merged[['count', 'area_km2']].corr().iloc[0,1]
print(f"Correlation between document count and country size in CCLW: {cclw_size_correlation.round(2)}")

cpdb_merged['area_km2'] = cpdb_merged['geometry'].to_crs('ESRI:54009').area / 1e6
cpdb_size_correlation = cpdb_merged[['count', 'area_km2']].corr().iloc[0,1]
print(f"Correlation between document count and country size in CPDB: {cpdb_size_correlation.round(2)}")